### Ingestion del archivo "movie_cast.json"

In [0]:
dbutils.widgets.text("p_environment", "")
v_environment = dbutils.widgets.get("p_environment")

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/commom_functions"

#### Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
moviecast_schema = StructType(fields=[
    StructField("movieId", IntegerType(), True),
    StructField("personId", IntegerType(), True),
    StructField("characterName", StringType(), True),
    StructField("genreId", IntegerType(), True),
    StructField("castOrder", IntegerType(), True)
    ])

In [0]:
moviescast_df = spark.read \
            .schema(moviecast_schema) \
            .option("multiLine", "true") \
            .json(f"{bronze_folder_path}/{v_file_date}/movie_cast.json")

In [0]:
display(moviescast_df)

#### Paso 2 - renombrar columnas y agregar columnas nuevas
1. renombrar "movieId" a "movie_id"
2. renombrar "personId" a "person_id"
3. renombrar "charaterName" a "character_name"
4. Agregar las columnas "ingestion_date" y "environment"

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
moviescast_with_columns_df = add_ingestion_date(moviescast_df) \
                            .withColumnsRenamed({"movieId": "movie_id"}) \
                            .withColumnsRenamed({"personId": "person_id"}) \
                            .withColumnsRenamed({"characterName": "character_name"}) \
                            .withColumn("environment", lit(v_environment)) \
                            .withColumn("file_date", lit(v_file_date))


In [0]:
display(moviescast_with_columns_df)

#### Paso 3 - Eliminar las columnas no necesarias

In [0]:
from pyspark.sql.functions import col

In [0]:
moviescast_final_df = moviescast_with_columns_df.drop(col("genreId")).drop(col("castOrder"))

In [0]:
display(moviescast_final_df)

#### Paso 4 - Escribir salida en un formato "Parquet"

In [0]:
# moviescast_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/movies_casts")

In [0]:
# display(spark.read.parquet("abfss://silver@moviehistory22.dfs.core.windows.net/movies_casts"))

In [0]:
# overwrite_partition("movie_silver", "movies_casts","file_date",v_file_date)

In [0]:
# moviescast_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.movies_casts")
# moviescast_final_df.write.mode("append").partitionBy("file_date").format("delta").saveAsTable("movie_silver.movies_casts")

In [0]:
merge_condition = 'tgt.movie_id = src.movie_id AND tgt.person_id= src.person_id AND tgt.file_date=src.file_date'
merge_delta_lake(moviescast_final_df, "movie_silver", "movies_casts", merge_condition, "file_date")

In [0]:
%sql
SELECT file_date, COUNT(1)
FROM movie_silver.movies_casts
GROUP BY file_date

In [0]:
%sql
SELECT * FROM movie_silver.movies_casts

In [0]:
dbutils.notebook.exit("Success")